<div id='top'>包括以下操作：</div>
<li><a href='#1'>位置编码</a></li>
<li><a href='#2'>pad填充 mask</a></li>
<li><a href='#3'></a></li>
<li><a href='#4'></a></li>
<li><a href='#5'></a></li>
<li><a href='#6'></a></li>

<div style='color:skyblue; font-size:24px' id='1'>位置编码</div>

<a href='#top'>▲ Top</a>

n_position=5：生成5个位置（位置0到4）的编码。</br>
d_model=4：每个位置编码的维度为4。</br>
输出矩阵：</br>
每行对应一个位置（例如第0行是位置0的编码）。</br>
奇偶维度交替使用正弦和余弦值（例如第0列是sin，第1列是cos，第2列是sin，第3列是cos）。</br>

扩展建议：</br>
尝试将 d_model 改为 512（Transformer标准维度），观察编码的平滑性。</br>
将 n_position 改为 100，验证长序列的位置编码是否仍满足相对位置关系。</br>

In [ ]:
import numpy as np
import torch

# 运用的公式都是
# 定义函数（与原文一致）
def get_sinusoid_encoding_table(n_position, d_model):
    def get_posi_angle_vec(position):
        for hid_j in range(d_model):
            print("hid_j:", hid_j)
        # 每一个 pos_i 对应一个位置向量, 位置向量的维度为 d_model
        # 以 pos_i 为行索引，hid_j 为列索引
        # 如 pos_i = 0, hid_j=0, 1, 2, 3， cal_angle(position, hid_j) 输入 0，0；0，1；0，2；0，3
        return [cal_angle(position, hid_j) for hid_j in range(d_model)] 
    def cal_angle(position, hid_idx):
        return position / np.power(10000, 2 * (hid_idx // 2) / d_model)
    for pos_i in range(n_position):
        print("pos_i:", pos_i)  # 输出位置索引
        print("get_posi_angle_vec(pos_i):", get_posi_angle_vec(pos_i)) # 输出位置向量
    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(n_position)]) # pos_i: 0,1,2,3,4
    '''
    "sinusoid_table: (位置向量)
        [[0.   0.   0.   0.  ]
        [1.   1.   0.01 0.01]
        [2.   2.   0.02 0.02]
        [3.   3.   0.03 0.03]
        [4.   4.   0.04 0.04]]
    '''


    '''
    # sinusoid_table[:, 0::2]:
        [[0.   0.  ]
        [1.   0.01]
        [2.   0.02]
        [3.   0.03]
        [4.   0.04]]
    '''
    # 从 0 开始，步长为 2，取出偶数维度的值
    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])  # 偶数维度用sin
    '''
    np.sin(sinusoid_table[:, 0::2]):
        [[ 0.          0.        ]
        [ 0.84147098  0.00999983]
        [ 0.90929743  0.01999867]
        [ 0.14112001  0.0299955 ]
        [-0.7568025   0.03998933]]
    '''

    '''
    sinusoid_table[:, 1::2]:
        [[0.   0.  ]
        [1.   0.01]
        [2.   0.02]
        [3.   0.03]
        [4.   0.04]]
    '''
    # 从 1 开始，步长为 2，取出奇数维度的值
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])  # 奇数维度用cos
    '''
    np.cos(sinusoid_table[:, 1::2]):
        [[ 1.          1.        ]
        [ 0.54030231  0.99995000]
        [-0.41614684  0.99980001]
        [-0.9899925   0.99955003]
        [-0.65364362  0.99920007]]
    '''

    return torch.FloatTensor(sinusoid_table)

# 指定输入参数
n_position = 5  # 位置数量（序列最大长度=5）
d_model = 4     # 编码维度（与词嵌入维度一致），每个位置的维度为4

# 运行函数
pos_encoding = get_sinusoid_encoding_table(n_position, d_model)

pos_i: 0
hid_j: 0
hid_j: 1
hid_j: 2
hid_j: 3
get_posi_angle_vec(pos_i): [0.0, 0.0, 0.0, 0.0]
pos_i: 1
hid_j: 0
hid_j: 1
hid_j: 2
hid_j: 3
get_posi_angle_vec(pos_i): [1.0, 1.0, 0.01, 0.01]
pos_i: 2
hid_j: 0
hid_j: 1
hid_j: 2
hid_j: 3
get_posi_angle_vec(pos_i): [2.0, 2.0, 0.02, 0.02]
pos_i: 3
hid_j: 0
hid_j: 1
hid_j: 2
hid_j: 3
get_posi_angle_vec(pos_i): [3.0, 3.0, 0.03, 0.03]
pos_i: 4
hid_j: 0
hid_j: 1
hid_j: 2
hid_j: 3
get_posi_angle_vec(pos_i): [4.0, 4.0, 0.04, 0.04]
hid_j: 0
hid_j: 1
hid_j: 2
hid_j: 3
hid_j: 0
hid_j: 1
hid_j: 2
hid_j: 3
hid_j: 0
hid_j: 1
hid_j: 2
hid_j: 3
hid_j: 0
hid_j: 1
hid_j: 2
hid_j: 3
hid_j: 0
hid_j: 1
hid_j: 2
hid_j: 3
0.8414709848078965


删除掉上面那些注释后的代码：

In [5]:
import numpy as np
import torch

# 运用的公式都是
# 定义函数（与原文一致）
def get_sinusoid_encoding_table(n_position, d_model):
    def get_posi_angle_vec(position):
        return [cal_angle(position, hid_j) for hid_j in range(d_model)] 
    def cal_angle(position, hid_idx):
        return position / np.power(10000, 2 * (hid_idx // 2) / d_model)
    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(n_position)]) # pos_i: 0,1,2,3,4
    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])  # 偶数维度用sin
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])  # 奇数维度用cos

    return torch.FloatTensor(sinusoid_table)

# 指定输入参数
n_position = 5  # 位置数量（序列最大长度=5）
d_model = 4     # 编码维度（与词嵌入维度一致），每个位置的维度为4

# 运行函数
pos_encoding = get_sinusoid_encoding_table(n_position, d_model)

# 查看输出
print("位置编码矩阵形状:", pos_encoding.shape)
print("\n编码内容:")
print(pos_encoding)

位置编码矩阵形状: torch.Size([5, 4])

编码内容:
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  0.9999],
        [ 0.9093, -0.4161,  0.0200,  0.9998],
        [ 0.1411, -0.9900,  0.0300,  0.9996],
        [-0.7568, -0.6536,  0.0400,  0.9992]])


<div style='color:skyblue; font-size:24px' id='2'>pad填充 mask</div>

<a href='#top'>▲ Top</a>

In [ ]:
import torch

# 构造输入张量
seq_q = torch.tensor([
    [1, 2, 0],   # 第一个句子（有效长度2，末尾填充0）
    [3, 0, 0]    # 第二个句子（有效长度1，末尾填充0）
])

seq_k = torch.tensor([
    [4, 5, 0, 0],  # 第一个句子（有效长度2，末尾填充两个0）
    [6, 7, 8, 0]   # 第二个句子（有效长度3，末尾填充一个0）
])

def get_attn_pad_mask(seq_q, seq_k):
    batch_size, len_q = seq_q.size()
    print("seq_q.size():", seq_q.size())
    print("batch_size, len_q:", batch_size, len_q)
    
    batch_size, len_k = seq_k.size()
    pad_attn_mask = seq_k.data.eq(0).unsqueeze(1)  # [batch_size, 1, len_k]
    return pad_attn_mask.expand(batch_size, len_q, len_k)  # [batch_size, len_q, len_k]

mask = get_attn_pad_mask(seq_q, seq_k)
print("Padding Mask Shape:", mask.shape)
print("Mask Values:\n", mask)

seq_q.size(): torch.Size([2, 3])
batch_size, len_q: 2 3
Padding Mask Shape: torch.Size([2, 3, 4])
Mask Values:
 tensor([[[False, False,  True,  True],
         [False, False,  True,  True],
         [False, False,  True,  True]],

        [[False, False, False,  True],
         [False, False, False,  True],
         [False, False, False,  True]]])
